# Gold Dimension — Date

Build the shared UrbanPulse calendar dimension.

This notebook:

1. Generates a fixed calendar range.
2. Creates standard date attributes.
3. Enriches dates with England and Wales bank holidays.
4. Validates dimensional keys.
5. Writes the Gold `dim_date` table.

**Source:** `workspace.urbanpulse_silver.bank_holidays`

**Target:** `workspace.urbanpulse_gold.dim_date`

**Grain:** One row per calendar date.

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import project components

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.dim_date import (
    generate_date_dimension,
    enrich_with_bank_holidays,
)

from urbanpulse.utils.config import (
    load_yaml,
)

## 3. Load date-dimension configuration

The calendar range is configuration-driven so it remains stable across pipeline executions.

In [0]:
CONFIG_PATH = (
    PROJECT_ROOT
    / "conf"
    / "project.yml"
)

config = load_yaml(
    str(CONFIG_PATH)
)

date_config = config[
    "date_dimension"
]

START_DATE = date_config[
    "start_date"
]

END_DATE = date_config[
    "end_date"
]

HOLIDAY_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "bank_holidays"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "dim_date"
)

print(
    f"Date range: "
    f"{START_DATE} → {END_DATE}"
)

print(
    f"Target: {TARGET_TABLE}"
)

## 4. Generate calendar dates

Generate one row per day across the configured date range.

In [0]:
date_df = generate_date_dimension(
    spark=spark,
    start_date=START_DATE,
    end_date=END_DATE,
)

print(
    f"Generated dates: "
    f"{date_df.count()}"
)

display(
    date_df
    .orderBy("calendar_date")
    .limit(10)
)

## 5. Verify generated calendar attributes

In [0]:
display(
    date_df.select(
        "date_key",
        "calendar_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend",
    )
    .orderBy(
        "calendar_date"
    )
    .limit(20)
)

## 6. Read Silver bank holidays

Bank-holiday information is joined by calendar date.

In [0]:
holidays_df = (
    spark.table(
        HOLIDAY_TABLE
    )
    .select(
        "holiday_date",
        "holiday_name",
    )
)

print(
    f"Silver holidays: "
    f"{holidays_df.count()}"
)

## 7. Add bank-holiday attributes

Dates matching one or more England and Wales holidays are marked as bank holidays.

If multiple holiday records share a date, their names are combined into one dimension value.

In [0]:
enriched_df = (
    enrich_with_bank_holidays(
        date_df=date_df,
        holidays_df=holidays_df,
    )
)

display(
    enriched_df
    .filter(
        F.col(
            "is_bank_holiday"
        )
    )
    .select(
        "date_key",
        "calendar_date",
        "day_name",
        "holiday_name",
    )
    .orderBy(
        "calendar_date"
    )
)

## 8. Add Gold metadata

Gold tables record when the dimensional dataset was built.

In [0]:
gold_df = (
    enriched_df
    .withColumn(
        "created_at",
        F.current_timestamp(),
    )
    .select(
        "date_key",
        "calendar_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend",
        "is_bank_holiday",
        "holiday_name",
        "created_at",
    )
)

## 9. Validate dimensional keys

Every calendar date must have exactly one non-null `date_key`.

In [0]:
total_rows = gold_df.count()

null_keys = (
    gold_df
    .filter(
        F.col("date_key").isNull()
    )
    .count()
)

duplicate_keys = (
    gold_df
    .groupBy("date_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_dates = (
    gold_df
    .groupBy("calendar_date")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(
    f"Rows: {total_rows}"
)

print(
    f"Null keys: {null_keys}"
)

print(
    f"Duplicate keys: {duplicate_keys}"
)

print(
    f"Duplicate dates: {duplicate_dates}"
)

if null_keys > 0:
    raise ValueError(
        "Null date keys detected."
    )

if duplicate_keys > 0:
    raise ValueError(
        "Duplicate date keys detected."
    )

if duplicate_dates > 0:
    raise ValueError(
        "Duplicate calendar dates detected."
    )

print(
    "Date dimension validation passed."
)

## 10. Validate weekend logic

Saturday and Sunday must be classified as weekends; weekdays must not.

In [0]:
invalid_weekends = (
    gold_df
    .filter(
        (
            F.col("day_name")
            .isin("Saturday", "Sunday")
        )
        !=
        F.col("is_weekend")
    )
)

if invalid_weekends.count() > 0:
    display(invalid_weekends)

    raise ValueError(
        "Weekend classification failed."
    )

print(
    "Weekend validation passed."
)

## 11. Write the Gold date dimension

`dim_date` is deterministic and inexpensive to regenerate, so the complete table is replaced on each successful build.

In [0]:
(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Created Gold table: "
    f"{TARGET_TABLE}"
)

## 12. Verify `dim_date`

In [0]:
%sql
SELECT
    date_key,
    calendar_date,
    year,
    quarter,
    month,
    month_name,
    week_of_year,
    day_of_week,
    day_name,
    is_weekend,
    is_bank_holiday,
    holiday_name
FROM workspace.urbanpulse_gold.dim_date
ORDER BY calendar_date
LIMIT 50;

In [0]:
%sql
SELECT
    COUNT(*) AS dates,
    MIN(calendar_date) AS first_date,
    MAX(calendar_date) AS last_date
FROM workspace.urbanpulse_gold.dim_date;

In [0]:
%sql
SELECT
    calendar_date,
    date_key
FROM workspace.urbanpulse_gold.dim_date
WHERE calendar_date IN (
    DATE '2026-01-01',
    DATE '2026-08-23',
    DATE '2030-12-31'
)
ORDER BY calendar_date;

In [0]:
%sql
SELECT
    calendar_date,
    day_name,
    holiday_name
FROM workspace.urbanpulse_gold.dim_date
WHERE is_bank_holiday = TRUE
ORDER BY calendar_date;

In [0]:
%sql
SELECT
    h.holiday_date,
    h.holiday_name
FROM workspace.urbanpulse_silver.bank_holidays h

LEFT ANTI JOIN
workspace.urbanpulse_gold.dim_date d

ON h.holiday_date = d.calendar_date;

In [0]:
%sql
SELECT
    day_of_week,
    day_name,
    COUNT(*) AS dates
FROM workspace.urbanpulse_gold.dim_date
GROUP BY
    day_of_week,
    day_name
ORDER BY day_of_week;

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.urbanpulse_gold.dim_date;